# Task 2.2 EDA — 스마트폰 리뷰 데이터 탐색

목적: 방법 선택 전에 데이터가 무엇을 갖고 있는지 파악 → **질문/가설 도출**

데이터: `si_dataset/review_for_analysis.json`  
- 총 2,757개 리뷰 / 7개 스마트폰 모델 (삼성 4종, 애플 3종)
- 필드: rating, helpfulCount, reviewSurveyAnswers, content, title, reviewAt, itemName, ...

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = Path('si_dataset/review_for_analysis.json')

In [ ]:
# JSON에 NaN, nullDeliberate 등 비표준 값이 포함되어 있어 전처리 후 로드
raw = DATA_PATH.read_text(encoding='utf-8')
raw = re.sub(r'\bNaN\b', 'null', raw)
raw = re.sub(r'\bnullDeliberate\b', 'null', raw)
records = json.loads(raw)

df = pd.json_normalize(records)
print(f'총 레코드: {len(df):,}개')
print(f'컬럼 수: {df.shape[1]}개')
df.head(2)

---
## 1. 기본 구조 확인

In [ ]:
# 분석에 쓸 핵심 컬럼만 정리
df['reviewAt_dt'] = pd.to_datetime(df['reviewAt'], unit='ms')
df['content_len'] = df['content'].fillna('').str.len()
df['brand'] = df['product_name'].apply(
    lambda x: 'Apple' if 'iphone' in str(x) else 'Samsung'
)

# 제품명 짧게
product_labels = {
    'iphone_17': 'iPhone 17',
    'iphone_17_pro': 'iPhone 17 Pro',
    'iphone_17_pro_max': 'iPhone 17 Pro Max',
    'galaxy_s26': 'Galaxy S26',
    'galaxy_s26_ultra': 'Galaxy S26 Ultra',
    'galaxy_z_fold7': 'Galaxy Z Fold7',
    'galaxy_z_flip7': 'Galaxy Z Flip7',
}
df['product_label'] = df['product_name'].map(product_labels)

print(df[['product_name', 'brand', 'rating', 'helpfulCount', 'content_len', 'reviewAt_dt']].dtypes)
df[['product_label', 'brand', 'rating', 'helpfulCount', 'helpfulTrueCount', 'helpfulFalseCount', 'content_len']].describe()

In [ ]:
# 결측치 현황
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('결측 있는 컬럼:')
print(missing.to_string())

---
## 2. 제품별 리뷰 수 & 평점 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2-1. 제품별 리뷰 수
product_order = df['product_label'].value_counts().index
colors = ['#4A90D9' if 'iPhone' in p else '#E85D5D' for p in product_order]
ax = axes[0]
df['product_label'].value_counts().reindex(product_order).plot(
    kind='bar', ax=ax, color=colors, edgecolor='white'
)
ax.set_title('제품별 리뷰 수', fontsize=13, fontweight='bold')
ax.set_xlabel('')
ax.set_xticklabels(product_order, rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

# 2-2. 전체 평점 분포
ax2 = axes[1]
rating_counts = df['rating'].value_counts().sort_index()
rating_pct = rating_counts / len(df) * 100
bars = ax2.bar(rating_counts.index, rating_counts.values,
               color=['#d9534f','#e88a3d','#f0c040','#5cb85c','#337ab7'],
               edgecolor='white')
ax2.set_title('전체 평점 분포', fontsize=13, fontweight='bold')
ax2.set_xlabel('별점')
ax2.set_xticks([1,2,3,4,5])
for bar, pct in zip(bars, rating_pct):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('output/eda_01_product_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n5점 비율: {rating_pct[5]:.1f}%  — 극단적인 쏠림 주의')

In [ ]:
# 제품별 평점 분포 히트맵
rating_by_product = (
    df.groupby(['product_label', 'rating'])
    .size()
    .unstack(fill_value=0)
)
# 비율로 변환
rating_by_product_pct = rating_by_product.div(rating_by_product.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    rating_by_product_pct,
    annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=0.5, ax=ax, cbar_kws={'label': '%'}
)
ax.set_title('제품별 평점 분포 (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('별점')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('output/eda_02_rating_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. 브랜드별 평점 비교 (Apple vs Samsung)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, brand in zip(axes, ['Apple', 'Samsung']):
    sub = df[df['brand'] == brand]['rating'].value_counts().sort_index()
    pct = sub / sub.sum() * 100
    color = '#4A90D9' if brand == 'Apple' else '#E85D5D'
    bars = ax.bar(sub.index, sub.values, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(f'{brand} — 평점 분포', fontsize=12, fontweight='bold')
    ax.set_xticks([1,2,3,4,5])
    ax.set_xlabel('별점')
    for bar, p in zip(bars, pct):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{p:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('브랜드별 평점 비교', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/eda_03_brand_rating.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('brand')['rating'].agg(['mean','median','std']).round(2))

---
## 4. 시계열: 리뷰 시점 분포

In [ ]:
df['review_month'] = df['reviewAt_dt'].dt.to_period('M')

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# 4-1. 월별 전체 리뷰 수
monthly = df.groupby('review_month').size()
ax1 = axes[0]
monthly.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='white')
ax1.set_title('월별 리뷰 수', fontsize=12, fontweight='bold')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=45)

# 4-2. 월별 브랜드별 리뷰 수
ax2 = axes[1]
monthly_brand = df.groupby(['review_month', 'brand']).size().unstack(fill_value=0)
monthly_brand.plot(kind='bar', ax=ax2, color=['#4A90D9', '#E85D5D'],
                   edgecolor='white', width=0.7)
ax2.set_title('월별 브랜드별 리뷰 수', fontsize=12, fontweight='bold')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(title='브랜드')

plt.tight_layout()
plt.savefig('output/eda_04_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

print('리뷰 기간:', df['reviewAt_dt'].min().date(), '~', df['reviewAt_dt'].max().date())

---
## 5. Helpful Votes 분석

In [ ]:
df_help = df[df['helpfulCount'] > 0].copy()
print(f'helpfulCount > 0 리뷰: {len(df_help)}개 ({len(df_help)/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 5-1. helpfulCount 분포 (log scale)
axes[0].hist(df_help['helpfulCount'], bins=30, color='teal', edgecolor='white')
axes[0].set_yscale('log')
axes[0].set_title('Helpful Count 분포 (log scale)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('도움돼요 수')

# 5-2. 평점별 평균 helpfulCount
avg_help_rating = df.groupby('rating')['helpfulCount'].mean()
axes[1].bar(avg_help_rating.index, avg_help_rating.values,
            color=['#d9534f','#e88a3d','#f0c040','#5cb85c','#337ab7'],
            edgecolor='white')
axes[1].set_title('평점별 평균 도움돼요', fontsize=11, fontweight='bold')
axes[1].set_xlabel('별점')
axes[1].set_xticks([1,2,3,4,5])

# 5-3. 리뷰 길이 vs helpfulCount (샘플)
axes[2].scatter(df_help['content_len'], df_help['helpfulCount'],
                alpha=0.3, s=15, color='purple')
axes[2].set_title('리뷰 길이 vs 도움돼요', fontsize=11, fontweight='bold')
axes[2].set_xlabel('리뷰 글자 수')
axes[2].set_ylabel('helpfulCount')

plt.tight_layout()
plt.savefig('output/eda_05_helpful.png', dpi=150, bbox_inches='tight')
plt.show()

# 가장 많은 도움돼요 받은 리뷰 TOP 5
print('\n--- 도움돼요 TOP 5 리뷰 ---')
top_help = df.nlargest(5, 'helpfulCount')[['product_label','rating','helpfulCount','title','content']]
for _, row in top_help.iterrows():
    print(f"[{row['product_label']} / ★{row['rating']} / 도움:{row['helpfulCount']}] {row['title']}")
    print(f"  {str(row['content'])[:100]}...\n")

---
## 6. reviewSurveyAnswers 분석 (가성비·디자인·카메라·무게·배터리)

In [ ]:
# survey 데이터 펼치기
survey_rows = []
for _, row in df.iterrows():
    answers = row.get('reviewSurveyAnswers')
    if not isinstance(answers, list):
        continue
    for ans in answers:
        if isinstance(ans, dict) and ans.get('question') and ans.get('answer'):
            survey_rows.append({
                'product_label': row['product_label'],
                'brand': row['brand'],
                'rating': row['rating'],
                'question': ans['question'],
                'answer': ans['answer'],
            })

sv = pd.DataFrame(survey_rows)
print(f'총 survey 응답: {len(sv):,}개')
print('\n질문 종류:')
print(sv['question'].value_counts().to_string())

In [ ]:
# 질문별 답변 분포
questions = sv['question'].value_counts().head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, q in zip(axes, questions):
    sub = sv[sv['question'] == q]['answer'].value_counts().head(8)
    sub.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(q, fontsize=11, fontweight='bold')
    ax.set_xlabel('응답 수')
    ax.invert_yaxis()

plt.suptitle('Survey 응답 분포 (질문별)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/eda_06_survey_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 브랜드별 Survey 답변 비교 — '가성비' 항목
q_target = '가성비'
sv_q = sv[sv['question'] == q_target]

brand_answer = (
    sv_q.groupby(['brand', 'answer'])
    .size()
    .unstack(fill_value=0)
)
# 비율
brand_answer_pct = brand_answer.div(brand_answer.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 4))
brand_answer_pct.T.plot(kind='bar', ax=ax,
                        color=['#4A90D9', '#E85D5D'],
                        edgecolor='white')
ax.set_title(f'브랜드별 [{q_target}] 응답 비율', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='브랜드')
plt.tight_layout()
plt.savefig('output/eda_07_survey_brand.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 평점 그룹별 Survey 답변 — 핵심 질문들
df['rating_group'] = pd.cut(df['rating'], bins=[0,2,3,5],
                             labels=['부정(1-2)', '중립(3)', '긍정(4-5)'])

# rating_group을 survey df에 merge
sv_merged = sv.merge(
    df[['reviewId', 'rating_group']],
    left_on=sv.index,
    right_on=df.reset_index().index,
    how='left'
)

print('\n--- 부정 리뷰(1-2점)에서 가장 많은 Survey 답변 TOP 10 ---')
neg_sv = sv[sv['rating'] <= 2]
print(neg_sv['answer'].value_counts().head(10).to_string())

print('\n--- 긍정 리뷰(4-5점)에서 가장 많은 Survey 답변 TOP 10 ---')
pos_sv = sv[sv['rating'] >= 4]
print(pos_sv['answer'].value_counts().head(10).to_string())

---
## 7. 리뷰 텍스트 기본 분석

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 7-1. 리뷰 길이 분포
axes[0].hist(df['content_len'].clip(upper=1500), bins=50,
             color='mediumpurple', edgecolor='white')
axes[0].set_title('리뷰 길이 분포 (글자 수)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('글자 수 (최대 1500 clip)')
axes[0].axvline(df['content_len'].median(), color='red', linestyle='--',
                label=f'중앙값 {df["content_len"].median():.0f}')
axes[0].legend()

# 7-2. 평점별 리뷰 길이 분포
df.boxplot(column='content_len', by='rating', ax=axes[1],
           showfliers=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
axes[1].set_title('평점별 리뷰 길이', fontsize=11, fontweight='bold')
axes[1].set_xlabel('별점')
axes[1].set_ylabel('글자 수')
plt.suptitle('')

plt.tight_layout()
plt.savefig('output/eda_08_content_len.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('rating')['content_len'].agg(['median','mean']).round(1))

In [ ]:
# 빈 / 너무 짧은 리뷰 현황
print(f"content가 null인 리뷰: {df['content'].isna().sum()}")
print(f"content 10글자 미만: {(df['content_len'] < 10).sum()}")
print(f"content 30글자 미만: {(df['content_len'] < 30).sum()}")

print('\n--- 짧은 리뷰 예시 (10글자 미만) ---')
short = df[df['content_len'] < 10][['product_label','rating','content']].head(10)
print(short.to_string())

---
## 8. 제품별 평균 평점 · 평균 리뷰 길이 종합

In [ ]:
summary = (
    df.groupby('product_label')
    .agg(
        리뷰수=('rating', 'count'),
        평균평점=('rating', 'mean'),
        중앙평점=('rating', 'median'),
        부정비율=('rating', lambda x: (x <= 2).mean() * 100),
        평균리뷰길이=('content_len', 'mean'),
        평균도움돼요=('helpfulCount', 'mean'),
    )
    .round(2)
    .sort_values('평균평점')
)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#4A90D9' if 'iPhone' in p else '#E85D5D' for p in summary.index]
bars = ax.bar(summary.index, summary['평균평점'], color=colors, edgecolor='white')
ax.set_ylim(3.5, 5.0)
ax.set_title('제품별 평균 평점', fontsize=12, fontweight='bold')
ax.set_ylabel('평균 별점')
ax.set_xticklabels(summary.index, rotation=30, ha='right')
for bar, val in zip(bars, summary['평균평점']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('output/eda_09_avg_rating.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 이상 패턴 / 흥미로운 지점 메모

아래 셀에서 직접 발견한 패턴을 텍스트로 정리하세요.

In [ ]:
# 부정 리뷰(1-2점) 중 helpfulCount 상위 리뷰 — 공감받은 불만
neg_reviews = df[df['rating'] <= 2].nlargest(10, 'helpfulCount')
print('=== 부정 리뷰 중 도움돼요 TOP 10 ===')
for _, row in neg_reviews.iterrows():
    print(f"[{row['product_label']} ★{row['rating']} 도움:{row['helpfulCount']}] {row['title']}")
    print(f"  {str(row['content'])[:150]}")
    print()

In [ ]:
# 출시 초기(첫 달) vs 이후 리뷰의 평점 변화
df_sorted = df.sort_values('reviewAt_dt')
first_date = df_sorted['reviewAt_dt'].min()

for product in df['product_label'].unique():
    sub = df[df['product_label'] == product].sort_values('reviewAt_dt')
    if len(sub) < 20:
        continue
    cutoff = sub['reviewAt_dt'].min() + pd.Timedelta(days=30)
    early = sub[sub['reviewAt_dt'] <= cutoff]['rating'].mean()
    later = sub[sub['reviewAt_dt'] > cutoff]['rating'].mean()
    n_early = (sub['reviewAt_dt'] <= cutoff).sum()
    n_later = (sub['reviewAt_dt'] > cutoff).sum()
    print(f"{product}: 초기 {early:.2f} (n={n_early}) → 이후 {later:.2f} (n={n_later})  diff={later-early:+.2f}")

---
## 10. 다음 단계: 질문/가설 후보

EDA에서 발견한 것들을 바탕으로 아래 형식으로 정리하세요.

```
Q1. ______ 인가?
  → 왜 흥미롭나: ______
  → 어떤 방법으로 답할 수 있나: ______
  → 비즈니스 가치: ______

Q2. ...
```

힌트로 쓸 수 있는 후보 방향들:
- 삼성 vs 애플 부정 리뷰의 **불만 카테고리가 구조적으로 다른가?**
- **도움돼요를 많이 받는 리뷰**는 어떤 내용을 담고 있나? (집단 공감 신호)
- 출시 직후 vs 장기 사용자 리뷰에서 **주제가 어떻게 바뀌나?**
- Survey 응답 vs 실제 별점 — **표면 만족과 내면 불만의 gap이 있는가?**
- 특정 모델에서만 반복 등장하는 **독특한 키워드/불만**은 무엇인가?